# Stochastic Calculus from Zero · Episode 25: Change of numéraire: the forward measure

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> **The short rate** (illustrative): starts at 3.85%, pulled towards 3.50% with mean reversion 0.30 and volatility 1% a year (Vasicek). Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Numéraires · 4. The option · 5. Three prices · 6. The forward is a martingale · 7. Why Black's formula works · 8. Part 3 recap · 9. Part 3 checkpoint · 10. Exercises · 11. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 25,
    "quantlib_version": ql.__version__,
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))

## 2. Last episode's answer

$P(\max_{t \le 1} W_t \ge 2) = 2(1 - N(2)) = 4.55\%$.

In [ ]:
out["prev"] = {"answer_pct": 2 * (1 - float(norm_cdf(2.0))) * 100}

## 3. Numéraires

A **numéraire** is a positive traded asset used as the unit of value. For each numéraire $N$ there is a measure $Q^N$ under which every traded price divided by $N$ is a martingale (Geman, El Karoui and Rochet, 1995):

$$\frac{V_t}{N_t} = E^{N}\Big[\frac{V_T}{N_T} \,\Big|\, \mathcal F_t\Big].$$

- $N$ = the bank account $B_t = e^{\int_0^t r\,ds}$: the risk-neutral measure $Q$ of Episode 21. Pricing needs $E^Q[e^{-\int_0^T r}\,V_T]$, a random discount factor inside the average.
- $N$ = the zero-coupon bond $P(t, T)$ maturing at $T$: the **$T$-forward measure** $Q^T$. Since $P(T, T) = 1$,

$$V_0 = P(0, T)\,E^{T}[V_T] ,$$

and the discount factor comes out of the average. The change of measure is $dQ^T/dQ = 1/(B_T\,P(0, T))$; by Girsanov, under $Q^T$ the Vasicek rate gains drift $-\sigma^2B(t, T)$, with $B(t, T) = (1 - e^{-a(T - t)})/a$.

## 4. The option

A European call, expiring in 1 year, on the zero-coupon bond maturing in 5 years. The strike is the **forward bond price** $P(0, 5)/P(0, 1)$, so the option is at the money forward.

In [ ]:
R0, THETA, A, SIG = 0.0385, 0.035, 0.30, 0.01
T1, T2 = 1.0, 5.0
P01, P05 = float(vasicek_bond(T1, R0, A, THETA, SIG)), float(vasicek_bond(T2, R0, A, THETA, SIG))
fwd = P05 / P01
K = fwd
vas = ql.Vasicek(R0, A, THETA, SIG)
ql_call = vas.discountBondOption(ql.Option.Call, K, T1, T2)
out["option"] = {"T1": T1, "T2": T2, "P01": P01, "P05": P05, "forward": fwd, "K": K, "ql": ql_call, "ql_bp": ql_call * 1e4,
                 "years": {"ger": 1995, "jamshidian": 1989}}
out["option"]

## 5. Three prices

**Risk-neutral Monte Carlo**: 100,000 paths of $r$ with exact daily steps, the integral by the trapezoid rule, payoff $e^{-\int_0^1 r}\max(P(1, 5) - K, 0)$ with $P(1, 5)$ from the closed form.

**Forward-measure Monte Carlo**: 100,000 paths of $r$ with the extra drift $-\sigma^2B(t, 1)$ (Euler, daily), payoff $P(0, 1)\max(P(1, 5) - K, 0)$: no path-wise discounting.

In [ ]:
rng = np.random.default_rng(seed)
n_paths, n = 100_000, 252
dt = T1 / n
r = np.full(n_paths, R0); integral = np.zeros(n_paths)
for _ in range(n):
    rn = ou_step(r, dt, A, THETA, SIG, rng.standard_normal(n_paths))
    integral += 0.5 * (r + rn) * dt
    r = rn
bond_rn = vasicek_bond(T2 - T1, r, A, THETA, SIG)
pay_rn = np.exp(-integral) * np.maximum(bond_rn - K, 0)

rf = np.full(n_paths, R0)
for i in range(n):
    B_t = (1 - math.exp(-A * (T1 - i * dt))) / A
    rf = rf + (A * (THETA - rf) - SIG ** 2 * B_t) * dt + SIG * math.sqrt(dt) * rng.standard_normal(n_paths)
bond_f = vasicek_bond(T2 - T1, rf, A, THETA, SIG)
pay_f = P01 * np.maximum(bond_f - K, 0)

three = {"paths": n_paths, "ql": ql_call, "rn": float(pay_rn.mean()), "rn_se": float(pay_rn.std() / math.sqrt(n_paths)),
         "fwd": float(pay_f.mean()), "fwd_se": float(pay_f.std() / math.sqrt(n_paths))}
for k in ("ql", "rn", "fwd", "rn_se", "fwd_se"):
    three[k + "_bp"] = three[k] * 1e4
assert abs(three["rn"] - ql_call) < 3 * three["rn_se"] and abs(three["fwd"] - ql_call) < 3 * three["fwd_se"] + 2e-5
out["three"] = three
pd.Series(three)

## 6. The forward is a martingale

$P(t, 5)/P(t, 1)$ is a traded price in units of the numéraire $P(t, 1)$, so it is a $Q^1$-martingale: its average at $t = 1$, where $P(1, 1) = 1$, is today's forward. Under the risk-neutral measure the matching statement needs the random discount factor: $E^Q[e^{-\int_0^1 r}P(1, 5)] = P(0, 5)$.

In [ ]:
mart = {"forward": fwd, "E_T": float(bond_f.mean()), "E_T_se": float(bond_f.std() / math.sqrt(n_paths)),
        "disc_Q": float((np.exp(-integral) * bond_rn).mean()), "P05": P05}
assert abs(mart["E_T"] - fwd) < 3 * mart["E_T_se"] + 2e-5
assert abs(mart["disc_Q"] - P05) < 1e-4
out["mart"] = mart
mart

## 7. Why Black's formula works

Under $Q^{T_1}$ the forward bond price $F_t = P(t, T_2)/P(t, T_1)$ is a martingale, and in Vasicek it is lognormal, with total variance

$$v^2 = \sigma^2 B(T_1, T_2)^2\,\frac{1 - e^{-2aT_1}}{2a}.$$

A lognormal martingale with known variance is exactly the setting of Black's formula (Series 3, Episode 21: caplets with Black-76):

$$C = P(0, T_1)\big(F_0 N(d_1) - K N(d_2)\big), \quad d_{1,2} = \frac{\ln(F_0/K) \pm \tfrac12 v^2}{v}.$$

This is Jamshidian's (1989) bond option formula. Caps work the same way: each forward rate is a martingale under its own forward measure, and assuming it lognormal gives Black-76 (normal gives Bachelier).

In [ ]:
B12 = (1 - math.exp(-A * (T2 - T1))) / A
v = SIG * B12 * math.sqrt((1 - math.exp(-2 * A * T1)) / (2 * A))
d1 = (math.log(fwd / K) + 0.5 * v * v) / v
black = P01 * (fwd * float(norm_cdf(d1)) - K * float(norm_cdf(d1 - v)))
assert abs(black - ql_call) < 1e-10
out["black"] = {"vol": v, "vol_pct": v * 100, "price": black, "price_bp": black * 1e4}
out["black"]

## 8. Part 3 recap

| Episode | Idea |
|---|---|
| 17 | SDEs; geometric Brownian motion solved |
| 18 | Mean reversion: Ornstein-Uhlenbeck and Vasicek |
| 19 | CIR and Feller's condition |
| 20 | Euler versus Milstein: strong and weak error |
| 21 | Girsanov: reweight paths, change the drift |
| 22 | Martingale representation: hedges exist |
| 23 | Feynman-Kac: expectations are PDEs |
| 24 | Reflection and barrier options |
| 25 | Change of numéraire: the forward measure |

## 9. Part 3 checkpoint

Try these before Part 4. The answers are in the cell below (and in the book).

1. Solve $dX = -aX\,dt + \sigma\,dW$ with $X_0 = x$. What are $E[X_t]$ and $\mathrm{Var}(X_t)$?
2. A share has drift 10%, volatility 30%, and the risk-free rate is 4%. What is the market price of risk, and the Girsanov weight $Z_T$?
3. Write the Feynman-Kac PDE for $u(t, x) = E[g(X_T) \mid X_t = x]$ with $dX = \kappa(m - X)\,dt + s\,dW$ and no discounting.
4. An up-and-in call and an up-and-out call have the same strike and barrier. What do their prices add up to?

In [ ]:
th = (0.10 - 0.04) / 0.30
out["checkpoint"] = {"q1": "E = x e^{-at}, Var = sigma^2 (1 - e^{-2at}) / (2a)", "q2_theta": th, "q2_half_theta_sq": 0.5 * th ** 2,
                     "q3": "u_t + kappa (m - x) u_x + 1/2 s^2 u_xx = 0, u(T, x) = g(x)", "q4": "the vanilla call"}
print(f"1. E[X_t] = x e^(-at), Var = sigma^2 (1 - e^(-2at)) / (2a)  (Episode 18)")
print(f"2. theta = (10% - 4%) / 30% = {th:.2f};  Z_T = exp(-{th:.2f} W_T - {0.5 * th ** 2:.2f} T)  (Episode 21)")
print("3. u_t + kappa (m - x) u_x + 1/2 s^2 u_xx = 0, u(T, x) = g(x)  (Episode 23)")
print("4. the vanilla call: on every path exactly one of them pays  (Episode 24)")

## 10. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Under $Q$ the bond has volatility $-\sigma B(t, T)$: $dP/P = r\,dt - \sigma B(t, T)\,dW$. Use Girsanov to show that under $Q^T$ the Vasicek rate has drift $a(\theta - r) - \sigma^2 B(t, T)$.

<details><summary>Answer</summary>

$Z_t = P(t, T)/(B_tP(0, T))$ is a $Q$-martingale with $dZ/Z = -\sigma B(t, T)\,dW$, so it is a Girsanov weight with $\theta_t = \sigma B(t, T)$: $W^T = W + \int_0^t \sigma B(s, T)\,ds$ is a $Q^T$-Brownian motion. Substitute $dW = dW^T - \sigma B\,dt$ into $dr = a(\theta - r)\,dt + \sigma\,dW$.

</details>

**Exercise 2 (By hand).** Why is $P(t, T_2)/P(t, T_1)$ a martingale under $Q^{T_1}$, and what is its value at $t = T_1$?

<details><summary>Answer</summary>

$P(t, T_2)$ is a traded price and $P(t, T_1)$ is the numéraire; prices in units of the numéraire are martingales. At $T_1$, $P(T_1, T_1) = 1$, so it equals $P(T_1, T_2)$, and its expectation is today's forward $P(0, T_2)/P(0, T_1)$.

</details>

**Exercise 3 (By hand).** Show put-call parity for bond options: $C - P = P(0, T_2) - K\,P(0, T_1)$.

<details><summary>Answer</summary>

$C - P$ pays $P(T_1, T_2) - K$ at $T_1$. Under $Q^{T_1}$: value $= P(0, T_1)\big(E^{T_1}[P(T_1, T_2)] - K\big) = P(0, T_1)(F_0 - K) = P(0, T_2) - KP(0, T_1)$.

</details>

**Exercise 4 (In code).** Price the matching put three ways: QuantLib, parity, and the forward-measure paths.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
ql_put = vas.discountBondOption(ql.Option.Put, K, T1, T2)
parity = ql_call - (P05 - K * P01)
mc_put = P01 * np.maximum(K - bond_f, 0)
se = mc_put.std() / math.sqrt(n_paths)
assert abs(ql_put - parity) < 1e-10 and abs(mc_put.mean() - ql_put) < 3 * se + 2e-5
print(f"put: QuantLib {ql_put * 1e4:.2f} bp, parity {parity * 1e4:.2f} bp, forward-measure Monte Carlo {mc_put.mean() * 1e4:.2f} ± {se * 1e4:.2f} bp")

**Exercise 5 (In code).** Use the 2-year forward measure: simulate to $t = 2$ with drift $a(\theta - r) - \sigma^2B(t, 2)$ and check that the average of $P(2, 5)$ is $P(0, 5)/P(0, 2)$.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
rng2 = np.random.default_rng(11)
x = np.full(50_000, R0); h = 2 / 504
for i in range(504):
    Bt = (1 - math.exp(-A * (2 - i * h))) / A
    x = x + (A * (THETA - x) - SIG ** 2 * Bt) * h + SIG * math.sqrt(h) * rng2.standard_normal(x.size)
b25 = vasicek_bond(3.0, x, A, THETA, SIG)
target = P05 / float(vasicek_bond(2.0, R0, A, THETA, SIG))
se = b25.std() / math.sqrt(x.size)
assert abs(b25.mean() - target) < 3 * se + 2e-5
print(f"E^2[P(2,5)] = {b25.mean():.5f} ± {se:.5f}; P(0,5)/P(0,2) = {target:.5f}")

## 11. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")